In [8]:
import collections

## Data ingestion pipeline
from langchain_core.documents import Document
import chromadb
from sqlalchemy.orm import persistence
from tomlkit import document

from tests.test_generate_golden_from_context import client

document = Document(
    page_content="This is a sample document for testing the data ingestion pipeline.",
    metadata={"source": "test_source", "author": "Bernardo Masoko", "date": "2024-06-15"}
)
document

Document(metadata={'source': 'test_source', 'author': 'Bernardo Masoko', 'date': '2024-06-15'}, page_content='This is a sample document for testing the data ingestion pipeline.')

In [14]:
# Create a simple .txt file in data directory
import os
os.makedirs('data', exist_ok=True)
with open('data/sample_document.txt', 'w') as f:
    f.write("This is a sample document for testing the data ingestion pipeline.\n")

In [18]:
# Load the document from the .txt file
from langchain_community.document_loaders import TextLoader

loader = TextLoader('..//data/privacysummary.pdf')
documents = loader.load()
# print(documents)

In [28]:
# Load the PDF document from the directory
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('..//data/privacysummary.pdf')
documents = loader.load()
print(type(documents[0]))
#documents

<class 'langchain_core.documents.base.Document'>


In [47]:
chunks = split_documents(documents, chunk_size=1000, chunk_overlap=200)
chunks

[Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'AdobePS5.dll Version 5.0.1', 'creationdate': 'D:20030513103711', 'title': 'Microsoft Word - MO02PBf_pdf.rtf', 'moddate': '2003-05-13T10:49:00-04:00', 'source': 'privacysummary.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1', 'author': 'Bernardo Masoko', 'chunk_start': 0, 'chunk_end': 109}, page_content='SUMMARY OF THE  \nHIPAA PRIVACY RULE \n \n \n \n \n \n \n \n \n \n \n \n \nHIPAA Compliance Assistance \n \nOCR PRIVACY BRIEF'),
 Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'AdobePS5.dll Version 5.0.1', 'creationdate': 'D:20030513103711', 'title': 'Microsoft Word - MO02PBf_pdf.rtf', 'moddate': '2003-05-13T10:49:00-04:00', 'source': 'privacysummary.pdf', 'total_pages': 25, 'page': 1, 'page_label': '2', 'author': 'Bernardo Masoko', 'chunk_start': 0, 'chunk_end': 1000}, page_content='i \nSUMMARY OF \nTHE HIPAA PRIVACY RULE \n \n \nContents \n \nIntroduction ...........

In [71]:
### embedding to vector store DB
import numpy
import chromadb
import uuid
# Create a collection in the ChromaDB client
client = chromadb.Client()
collection = client.get_or_create_collection(name="my_collection", metadata={"description": "A collection of documents for testing."})

for i, chunk in enumerate(chunks):
    # Create an embedding for the chunk
    embedding = numpy.random.rand(768).tolist()  # Replace with actual embedding generation
    # Add the chunk and its embedding to the collection
    collection.add(
        documents=[chunk.page_content],
        metadatas=[chunk.metadata],
        embeddings=[embedding],
        ids=[f"chunk_{i}"]
    )
    print(f"Added chunk {i} to the collection with ID: chunk_{i}")

Added chunk 0 to the collection with ID: chunk_0
Added chunk 1 to the collection with ID: chunk_1
Added chunk 2 to the collection with ID: chunk_2
Added chunk 3 to the collection with ID: chunk_3
Added chunk 4 to the collection with ID: chunk_4
Added chunk 5 to the collection with ID: chunk_5
Added chunk 6 to the collection with ID: chunk_6
Added chunk 7 to the collection with ID: chunk_7
Added chunk 8 to the collection with ID: chunk_8
Added chunk 9 to the collection with ID: chunk_9
Added chunk 10 to the collection with ID: chunk_10
Added chunk 11 to the collection with ID: chunk_11
Added chunk 12 to the collection with ID: chunk_12
Added chunk 13 to the collection with ID: chunk_13
Added chunk 14 to the collection with ID: chunk_14
Added chunk 15 to the collection with ID: chunk_15
Added chunk 16 to the collection with ID: chunk_16
Added chunk 17 to the collection with ID: chunk_17
Added chunk 18 to the collection with ID: chunk_18
Added chunk 19 to the collection with ID: chunk_19


In [73]:
# Query the collection to retrieve documents based on a query embedding
query_embedding = numpy.random.rand(768).tolist()  # Replace with actual query embedding generation
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)
print("Query Results:")
for result in results['documents'][0]:
    print(result)

Query Results:
, unless it can 
specifically justify the whole record as the amount reasonably needed for the purpose.  
See OCR “Minimum Necessary” Guidance. 
 
The minimum necessary requirement is not imposed in any of the following 
circumstances:  (a) disclosure to or a request by a health care provider for treatment; 
(b) disclosure to an individual who is the subject of the information, or the 
individual’s personal representative; (c) use or disclosure made pursuant to an 
authorization; (d) disclosure to HHS for complaint investigation, compliance review 
or enforcement; (e) use or disclosure that is required by law; or (f) use or disclosure 
required for compliance with the HIPAA Transactions Rule or other HIPAA 
Administrative Simplification Rules. 
 
Access and Uses.  For internal uses, a covered entity must develop and implement 
policies and procedures that restrict access and uses of protected health information 
based on the specific roles of the members of their workfor